# Zero-Shot Learning für die Spamerkennung

**Zero-Shot Learning** ist ein Setup, in dem ein Modell Beispiele in Kategorien klassifiziert, auf die es nie explizit trainiert wurde. Ein großes Sprachmodell wird auf sehr viel Text vortrainiert und kann bei der Inferenz die Aufgabe bekommen, neuen Text in Labels einzuordnen, die es während des Trainings nie gesehen hat. Dieselbe Idee lässt sich auf Text, Bilder und andere Bereiche anwenden.

In Notebook 1 haben wir einen Klassifikator auf Spamdaten mit Labels trainiert. Hier gehen wir den umgekehrten Weg: Wir nutzen einen vortrainierten Transformer ohne eigenes Training und lassen ihn Nachrichten als Spam oder Nicht-Spam kennzeichnen.

## Lernziele

Am Ende dieses Notebooks kannst du:

- eine Pipeline für die Zero-Shot-Klassifikation mit der Transformers-Bibliothek erstellen.
- ein vortrainiertes Modell anwenden, um Nachrichten ohne aufgabenspezifisches Training mit Labels zu versehen.
- die Vorhersagen mit dem Hamming-Loss evaluieren und mit einem trainierten Klassifikator vergleichen.

## Bibliotheken und Daten einrichten

In [ ]:
from transformers import pipeline
import pandas as pd
from tqdm import tqdm

In [ ]:
data = pd.read_csv(
    "data/SMSSpamCollection.txt",
    encoding="utf-8",
    header=None,
    delimiter="\t",
    names=["target", "text"],
)

In [ ]:
data.head(5)

In [ ]:
print(f"There are {data.shape[0]} rows in the dataset")

## Zero-Shot-Spamklassifikation

Die Funktion `pipeline` aus der Transformers-Bibliothek fasst den Download des Modells, die Tokenisierung und die Inferenz in einem einzigen Objekt zusammen. Für die Zero-Shot-Klassifikation nutzt sie ein Modell, das auf Natural Language Inference trainiert wurde. Damit können wir bei der Inferenz beliebige Kandidatenlabels übergeben. Wir geben ihr den Text der Nachricht und unsere Kandidatenlabels, und sie gibt für jedes Label einen Score zurück.

### Pipeline mit einer Zeile Code vorbereiten

In [ ]:
classifier = pipeline(
    "zero-shot-classification", device=0
)  # device=0 nutzt die erste GPU; setze device=-1, um auf der CPU zu rechnen

Standardmäßig lädt die Zero-Shot-Pipeline [**facebook/bart-large-mnli**](https://huggingface.co/facebook/bart-large-mnli), ein BART-Modell, das auf dem Datensatz MultiNLI für Natural Language Inference feinabgestimmt wurde. Weil es darauf trainiert wurde zu beurteilen, ob ein Satz einen anderen impliziert, kann es jedes Kandidatenlabel, das wir angeben, als Hypothese bewerten. Genau das macht die Zero-Shot-Klassifikation möglich.

Mehr dazu findest du in der [Model Card](https://huggingface.co/facebook/bart-large-mnli) und im [BART-Paper](https://arxiv.org/pdf/1910.13461).

### Vorhersagen treffen

Das Modell funktioniert am besten mit **aussagekräftigen Labels**. Die rohen Tags `spam` und `ham` haben für ein Allzweckmodell wenig Bedeutung, deshalb bilden wir sie auf beschreibendere Wendungen ab: `click bait` und `written by humans`. Der genaue Hamming-Loss hängt von der Version des Modells ab, aber beschreibendere Labels geben dem Modell in der Regel eine bessere Chance.

Findest du Label-Beschreibungen, die den Hamming-Loss weiter senken?

In [ ]:
category_map = {"spam": "click bait", "ham": "written by humans"}

In [ ]:
candidate_labels = list(category_map.values())
predictedCategories = []
trueCategories = []
for i in tqdm(range(100)):
    text = data.iloc[i,]["text"]
    cat = [data.iloc[i,]["target"]]
    res = classifier(text, candidate_labels, multi_label=False)
    labels = res["labels"]
    scores = res["scores"]  # die zu den Labels gehörenden Scores extrahieren
    res_dict = {label: score for label, score in zip(labels, scores)}
    sorted_dict = dict(
        sorted(res_dict.items(), key=lambda x: x[1], reverse=True)
    )  # das Dictionary der Labels absteigend nach ihrem Score sortieren
    categories = next(k for i, (k, v) in enumerate(sorted_dict.items()))

    predictedCategories.append(categories)
    trueCats = [category_map[x] for x in cat]
    trueCategories.append(trueCats)

In [ ]:
for y_true, y_pred in zip(trueCategories[:3], predictedCategories[:3]):
    print(f"True Categories {y_true}")
    print(f"Predicted Categories {y_pred}")
    print("#" * 50)

### Hamming-Loss
Der Hamming-Loss ist der Anteil der Labels, die falsch vorhergesagt werden.

In [ ]:
from sklearn.metrics import hamming_loss

print(
    f"The hamming loss is {hamming_loss(trueCategories, predictedCategories):.4f} compared to 0.0237 from the last trained model in Notebook 1"
)

**Ergebnis:** Mit den Labels `click bait` und `written by humans` liegt der Hamming-Loss für die ersten 100 Nachrichten bei etwa 0,35, also wird ungefähr ein Drittel der Nachrichten falsch klassifiziert. Das ist viel schlechter als der trainierte Klassifikator in Notebook 1, dessen Hamming-Loss bei etwa 0,02 lag.

Um genau diesen Trade-off geht es in diesem Notebook. Die Zero-Shot-Klassifikation braucht keine Daten mit Labels und keinen Trainingsschritt. Deshalb ist sie nützlich für schnelles Prototyping oder wenn du überhaupt keine Labels hast. Ein kleines Modell, das direkt auf die Aufgabe trainiert wurde, liegt bei der Accuracy trotzdem vorn. Zero-Shot ist also ein praktischer Ausgangspunkt, aber kein kostenloser Ersatz für einen trainierten Klassifikator.

## Prüfe dein Verständnis

Findest du bessere „Labels“ für die Beschreibung der Kategorien, um den Hamming-Loss zu verbessern?

### Du bist dran: bessere Labels entwerfen

Die beschreibenden Labels haben den Hamming-Loss gesenkt, aber wahrscheinlich findest du noch bessere. Vergleiche ein paar Sets von Kandidatenlabels und miss, welches gewinnt.

**Schritte**
1. Definiere 2 oder 3 Label-Zuordnungen, zum Beispiel `{"spam": "advertising", "ham": "personal message"}`.
2. Klassifiziere für jede Zuordnung die ersten 100 Nachrichten (verwende die Schleife von oben wieder) und berechne den Hamming-Loss.
3. Gib den Hamming-Loss für jedes Label-Set aus und notiere, welches am besten ist.

**Tipp:** Verpacke die Vorhersageschleife von vorhin in eine Funktion `evaluate_labels(category_map)`, die den Hamming-Loss zurückgibt, und rufe sie dann für jedes Set von Kandidatenlabels auf.

**Erwartetes Ergebnis:** Der Hamming-Loss unterscheidet sich je nach Label-Set; konkretere, beschreibende Labels erzielen meist einen niedrigeren (besseren) Score.

In [ ]:
from sklearn.metrics import hamming_loss


def evaluate_labels(category_map):
    labels = list(category_map.values())
    predicted, true = [], []
    # TODO 1: über die ersten 100 Nachrichten iterieren, jede mit `labels` klassifizieren
    # und das vorhergesagte und das tatsächliche Label sammeln (siehe die Schleife oben)
    ...
    # TODO 2: den Hamming-Loss zwischen true und predicted zurückgeben
    return ...


# TODO 3: ein paar Label-Zuordnungen ausprobieren und für jede den Hamming-Loss ausgeben
label_sets = {
    "clickbait/human": {"spam": "click bait", "ham": "written by humans"},
    # füge eigene hinzu, z. B. {"spam": "advertising", "ham": "personal message"}
}
for name, cmap in label_sets.items():
    print(name, "->", evaluate_labels(cmap))

## Zusammenfassung

In diesem Notebook hast du:

- eine Pipeline für die Zero-Shot-Klassifikation mit einem vortrainierten Transformer erstellt.
- sie angewendet, um SMS ohne aufgabenspezifisches Training mit Labels zu versehen.
- die Vorhersagen mit dem Hamming-Loss evaluiert und mit dem trainierten Klassifikator aus Notebook 1 verglichen.

Die Zero-Shot-Klassifikation ist praktisch, wenn Daten mit Labels knapp sind, aber ein Modell, das auf die Aufgabe trainiert wurde, liefert hier eine bessere Accuracy.

## Quellen & weiterführende Literatur

- [**Hugging Face: Zero-Shot Classification**](https://huggingface.co/tasks/zero-shot-classification): Was die Zero-Shot-Klassifikation ist und welche Modelle sich dafür eignen.
- [**Hugging Face: Pipelines**](https://huggingface.co/docs/transformers/main_classes/pipelines): Referenz zur hier verwendeten `pipeline`-API.